# Notebook 87 — Paired HITL Development Draft, Blind Review, Freeze, and A1 Cleanup Truth Experiment

**Purpose:** take the existing paired annotated pages through the same YOLO draft/confidence-map and human review workflow used for production HITL, while keeping the paired-development population physically and logically separate from the real-world HITL population and keeping clean truth inaccessible until the reviewed-mask and observable-output locks are closed.

## Population separation

- Real-world HITL remains under `human_in_the_middle/` and is never written by this notebook.
- Paired annotated pages are read from `source/annotated/`.
- Clean pages remain under `source/clean/` and are not resolved until the truth-scoring stage.
- All draft, confidence, review, audit and lock artefacts are written under `paired_hitl_development/`.

## Predeclared cleanup candidates

All candidates start from **A1**, where V7.8 `AMBIGUOUS` pixels retain exact observed bytes. Additional cleanup is permitted only at A1-preserved AMBIGUOUS pixels:

- `A1_BASELINE`: no additional cleanup.
- `C090_P010`: `p(annotation) >= 0.900` and `p_print < 0.10` → paper.
- `C0875_P010`: `p(annotation) >= 0.875` and `p_print < 0.10` → paper.
- `C085_P010`: `p(annotation) >= 0.850` and `p_print < 0.10` → paper.

No threshold selection or promotion is performed automatically.

In [ ]:
# ============================================================
# 0. Mount Drive and install runtime dependencies
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics gradio

In [ ]:
# ============================================================
# 1. Configuration and imports
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from dataclasses import asdict
import hashlib, json, shutil, gc, random, re

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# Existing paired data. CLEAN_DIR is intentionally not used
# before the truth-scoring stage.
ANNOTATED_DIR = ROOT / 'source' / 'annotated'
CLEAN_DIR = ROOT / 'source' / 'clean'

# Completely separate experimental HITL namespace.
PH = ROOT / 'paired_hitl_development'
DRAFT_MASK_DIR = PH / 'draft_masks'
CONFIDENCE_DIR = PH / 'confidence_maps'
DRAFT_AUDIT_DIR = PH / 'draft_audit'
REVIEWED_MASK_DIR = PH / 'reviewed_masks'
REVIEW_AUDIT_DIR = PH / 'audit'
LOCK_DIR = PH / 'locks'
RUNS_DIR = PH / 'runs'

for folder in [PH, DRAFT_MASK_DIR, CONFIDENCE_DIR, DRAFT_AUDIT_DIR,
               REVIEWED_MASK_DIR, REVIEW_AUDIT_DIR, LOCK_DIR, RUNS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ROOT / 'training_runs' / '20261006_150912_yolo11n-seg' / 'weights' / 'best.pt'
V89_OPERATOR_PATH = ROOT / 'v8_9_release' / 'v8_9_operator.py'
P_PRINT_BUNDLE = ROOT / 'p_print_provider_v8_2f'
REMOVE_BUNDLE = ROOT / 'notebook52f_remove_bundle'

# Override only if your final paths differ.
SOURCE_GROUP_MANIFEST_CANDIDATES = [
    ROOT / 'development_score_manifest.csv',
    ROOT / 'tables' / 'development_score_manifest.csv',
]

PREDICTION_CONFIDENCE = 0.20
TILE_SIZE = 640
TILE_STRIDE = 320
MODEL_IMAGE_SIZE = 640
MINIMUM_DRAFT_COMPONENT_AREA = 10
FORCE_REBUILD_DRAFTS = False

REVIEWER_CODE = 'DB'  # change before review
RANDOM_SEED = 42
SUPPORTED = {'.jpg','.jpeg','.png','.tif','.tiff','.bmp'}

CANDIDATES = {
    'A1_BASELINE': None,
    'C090_P010': (0.900, 0.10),
    'C0875_P010': (0.875, 0.10),
    'C085_P010': (0.850, 0.10),
}

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_notebook87')
RUN = RUNS_DIR / RUN_ID
for sub in ['tables','audit','pages','visuals','upload']:
    (RUN / sub).mkdir(parents=True, exist_ok=True)


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


def md5_file(path, chunk=1024*1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


def image_files(folder):
    return sorted(p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED)


def read_gray(path):
    a = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    require(a is not None, f'Unable to read image: {path}')
    return a

print('Run:', RUN)
print('CUDA:', torch.cuda.is_available())

## Stage A — Generate production-style drafts and confidence maps

This stage uses the same frozen YOLO checkpoint, threshold, tile size and overlap as the production HITL draft generator, but writes only to `paired_hitl_development/`.

In [ ]:
# ============================================================
# 2. Build paired annotated roster without touching clean truth
# ============================================================

annotated_map = {p.stem: p for p in image_files(ANNOTATED_DIR)}
require(annotated_map, f'No paired annotated pages found in {ANNOTATED_DIR}')

manifest_path = next((p for p in SOURCE_GROUP_MANIFEST_CANDIDATES if p.exists()), None)
manifest = pd.read_csv(manifest_path, dtype=str).fillna('') if manifest_path else pd.DataFrame()
manifest_by_page = manifest.set_index('page') if len(manifest) and 'page' in manifest.columns else None

roster_rows = []
for page, source in sorted(annotated_map.items()):
    source_group = ''
    if manifest_by_page is not None and page in manifest_by_page.index and 'source_group' in manifest_by_page.columns:
        source_group = str(manifest_by_page.loc[page, 'source_group'])
    roster_rows.append({
        'page': page,
        'page_key': f'paired_hitl_development__{page}',
        'population_role': 'PAIRED_HITL_DEVELOPMENT',
        'source_group': source_group,
        'annotated_path': str(source),
        'annotated_md5': md5_file(source),
    })

roster = pd.DataFrame(roster_rows)
roster.to_csv(RUN / 'tables' / 'paired_annotated_roster.csv', index=False)
print('Paired annotated pages:', len(roster))
display(roster.head())

In [ ]:
# ============================================================
# 3. Frozen YOLO confidence-map generator
# ============================================================

require(MODEL_PATH.exists(), f'Model not found: {MODEL_PATH}')
require(torch.cuda.is_available(), 'Select a T4 GPU runtime for draft generation.')

MODEL = YOLO(str(MODEL_PATH))
MODEL_MD5 = md5_file(MODEL_PATH)


def tile_positions(n):
    if n <= TILE_SIZE:
        return [0]
    p = list(range(0, n - TILE_SIZE + 1, TILE_STRIDE))
    if p[-1] != n - TILE_SIZE:
        p.append(n - TILE_SIZE)
    return p


def confidence_map(gray):
    colour = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    h, w = gray.shape
    ph, pw = max(h, TILE_SIZE), max(w, TILE_SIZE)
    canvas = np.full((ph, pw, 3), 255, np.uint8)
    canvas[:h, :w] = colour
    conf = np.zeros((ph, pw), np.float32)

    for y in tile_positions(ph):
        for x in tile_positions(pw):
            result = MODEL.predict(
                canvas[y:y+TILE_SIZE, x:x+TILE_SIZE],
                conf=PREDICTION_CONFIDENCE,
                imgsz=MODEL_IMAGE_SIZE,
                device=0,
                verbose=False,
                retina_masks=True,
            )[0]
            if result.masks is None:
                continue
            masks = result.masks.data.cpu().numpy()
            scores = result.boxes.conf.cpu().numpy()
            for mask, score in zip(masks, scores):
                mask = cv2.resize(mask, (TILE_SIZE, TILE_SIZE), interpolation=cv2.INTER_NEAREST) > 0.5
                region = conf[y:y+TILE_SIZE, x:x+TILE_SIZE]
                use = mask & (score > region)
                region[use] = score

    return conf[:h, :w]


def remove_small_components(mask, minimum_area=MINIMUM_DRAFT_COMPONENT_AREA):
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    out = np.zeros(mask.shape, np.uint8)
    for label in range(1, n):
        if int(stats[label, cv2.CC_STAT_AREA]) >= minimum_area:
            out[labels == label] = 1
    return out

In [ ]:
# ============================================================
# 4. Generate/resume paired drafts and confidence maps
# ============================================================

generation_rows = []

for _, r in roster.iterrows():
    page = str(r.page)
    source = Path(r.annotated_path)
    source_md5 = str(r.annotated_md5)

    mask_path = DRAFT_MASK_DIR / f'{page}_draft.png'
    conf_path = CONFIDENCE_DIR / f'{page}_confidence.npy'
    audit_path = DRAFT_AUDIT_DIR / f'{page}.json'

    current = False
    if not FORCE_REBUILD_DRAFTS and mask_path.exists() and conf_path.exists() and audit_path.exists():
        try:
            old = json.loads(audit_path.read_text())
            current = (
                old.get('source_md5') == source_md5 and
                old.get('model_md5') == MODEL_MD5 and
                float(old.get('prediction_confidence')) == PREDICTION_CONFIDENCE
            )
        except Exception:
            current = False

    if current:
        status = 'REUSED'
    else:
        gray = read_gray(source)
        conf = confidence_map(gray)
        binary = conf >= PREDICTION_CONFIDENCE
        binary = remove_small_components(binary)
        np.save(conf_path, conf.astype(np.float16))
        require(cv2.imwrite(str(mask_path), binary.astype(np.uint8) * 255), f'{page}: write draft failed')
        audit = {
            'created_utc': datetime.now(timezone.utc).isoformat(),
            'population_role': 'PAIRED_HITL_DEVELOPMENT',
            'clean_truth_available_to_reviewer': False,
            'source_file': str(source),
            'source_md5': source_md5,
            'model_path': str(MODEL_PATH),
            'model_md5': MODEL_MD5,
            'prediction_confidence': PREDICTION_CONFIDENCE,
            'minimum_component_area': MINIMUM_DRAFT_COMPONENT_AREA,
            'draft_pixels': int(binary.sum()),
            'confidence_map': str(conf_path),
            'draft_mask': str(mask_path),
            'shape': list(gray.shape),
        }
        audit_path.write_text(json.dumps(audit, indent=2))
        status = 'GENERATED'

    generation_rows.append({
        'page': page,
        'status': status,
        'draft_mask': str(mask_path),
        'confidence_map': str(conf_path),
        'draft_audit': str(audit_path),
    })
    print(page, status)

pd.DataFrame(generation_rows).to_csv(RUN / 'tables' / 'draft_generation_summary.csv', index=False)

## Stage B — Blind paired HITL review

The review interface below reads only the annotated page, draft mask and continuous detector-confidence map. It never resolves `source/clean`.

### Controls

- **Brush add:** paint pixels into the authoritative reviewed mask.
- **Brush erase:** erase pixels from the authoritative reviewed mask.
- **Confidence threshold display:** changes only the visual overlay, not the saved reviewed mask.
- **Save & next:** writes the reviewed mask and audit record under `paired_hitl_development/`.

If the Gradio version in Colab is incompatible with the canvas editor, export the displayed page/mask and use the existing Notebook 07 reviewer pointed at `paired_hitl_development/`; the namespace and audit contract remain the same.

In [ ]:
# ============================================================
# 5. Build review queue and helper visualisations
# ============================================================

review_rows = []
for _, r in roster.iterrows():
    page = str(r.page)
    reviewed_path = REVIEWED_MASK_DIR / f'{page}_reviewed.png'
    review_audit = REVIEW_AUDIT_DIR / f'{page}.json'
    review_rows.append({
        **r.to_dict(),
        'draft_mask_path': str(DRAFT_MASK_DIR / f'{page}_draft.png'),
        'confidence_path': str(CONFIDENCE_DIR / f'{page}_confidence.npy'),
        'reviewed_mask_path': str(reviewed_path),
        'review_audit_path': str(review_audit),
        'reviewed': bool(reviewed_path.exists() and review_audit.exists()),
    })

review_queue = pd.DataFrame(review_rows)
review_queue.to_csv(RUN / 'tables' / 'review_queue.csv', index=False)
print('Already reviewed:', int(review_queue.reviewed.sum()), '/', len(review_queue))


def confidence_overlay(gray, conf, threshold=0.20, alpha=0.45):
    base = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    conf = np.asarray(conf, np.float32)
    visible = conf >= threshold
    colour = np.zeros_like(base)
    # Rainbow: blue -> cyan -> green -> yellow -> red.
    heat = cv2.applyColorMap(np.clip(conf * 255, 0, 255).astype(np.uint8), cv2.COLORMAP_TURBO)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    out = base.copy()
    out[visible] = np.clip((1-alpha)*base[visible] + alpha*heat[visible], 0, 255).astype(np.uint8)
    return out


def mask_overlay(gray, mask, colour=(255,0,0), alpha=0.38):
    base = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    out = base.copy()
    m = np.asarray(mask, bool)
    colour_arr = np.array(colour, np.float32)
    out[m] = np.clip((1-alpha)*base[m] + alpha*colour_arr, 0, 255).astype(np.uint8)
    return out

In [ ]:
# ============================================================
# 6. Launch blind review interface
# ============================================================

import gradio as gr

pending = review_queue[~review_queue.reviewed].copy().reset_index(drop=True)
review_state = {'index': 0, 'mask': None}


def load_review_item(index):
    if index >= len(pending):
        return None, None, None, 'Review complete.'
    row = pending.iloc[index]
    gray = read_gray(row.annotated_path)
    draft = read_gray(row.draft_mask_path) > 0
    conf = np.load(row.confidence_path).astype(np.float32)
    require(gray.shape == draft.shape == conf.shape, f'{row.page}: review shape mismatch')
    review_state['index'] = index
    review_state['mask'] = draft.astype(np.uint8) * 255
    return (
        gray,
        confidence_overlay(gray, conf, 0.20),
        mask_overlay(gray, draft),
        f'{index+1}/{len(pending)} — {row.page}',
    )


def update_threshold(value):
    index = review_state['index']
    if index >= len(pending):
        return None
    row = pending.iloc[index]
    gray = read_gray(row.annotated_path)
    conf = np.load(row.confidence_path).astype(np.float32)
    return confidence_overlay(gray, conf, float(value))


def save_and_next(mask_image, notes):
    index = review_state['index']
    require(index < len(pending), 'No pending review item.')
    row = pending.iloc[index]
    gray = read_gray(row.annotated_path)

    # Gradio image editors may supply RGB/RGBA or a dict. Normalize fail-closed.
    value = mask_image
    if isinstance(value, dict):
        value = value.get('composite', value.get('background', value.get('layers', [None])[-1]))
    require(value is not None, 'Reviewer mask is empty.')
    value = np.asarray(value)
    if value.ndim == 3:
        value = cv2.cvtColor(value.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    mask = value > 127
    require(mask.shape == gray.shape, f'{row.page}: reviewed mask shape mismatch')

    out_path = Path(row.reviewed_mask_path)
    audit_path = Path(row.review_audit_path)
    require(cv2.imwrite(str(out_path), mask.astype(np.uint8)*255), f'{row.page}: reviewed mask write failed')

    draft = read_gray(row.draft_mask_path) > 0
    added = mask & ~draft
    erased = draft & ~mask
    retained = mask & draft

    audit = {
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'population_role': 'PAIRED_HITL_DEVELOPMENT',
        'clean_truth_available_to_reviewer': False,
        'reviewer_code': REVIEWER_CODE,
        'source_file': str(row.annotated_path),
        'source_md5': str(row.annotated_md5),
        'draft_mask': str(row.draft_mask_path),
        'draft_mask_sha256': sha256_file(row.draft_mask_path),
        'confidence_map': str(row.confidence_path),
        'confidence_map_sha256': sha256_file(row.confidence_path),
        'corrected_mask_output': str(out_path),
        'corrected_mask_sha256': sha256_file(out_path),
        'draft_pixels': int(draft.sum()),
        'reviewed_pixels': int(mask.sum()),
        'pixels_added_by_reviewer': int(added.sum()),
        'pixels_erased_by_reviewer': int(erased.sum()),
        'draft_pixels_retained': int(retained.sum()),
        'notes': str(notes or ''),
    }
    audit_path.write_text(json.dumps(audit, indent=2))

    review_state['index'] += 1
    return (*load_review_item(review_state['index']), '')


with gr.Blocks() as demo:
    gr.Markdown('# Paired HITL Development Reviewer — clean truth unavailable')
    status = gr.Markdown()
    with gr.Row():
        observed_view = gr.Image(label='Annotated page', interactive=False)
        confidence_view = gr.Image(label='YOLO confidence overlay', interactive=False)
        mask_editor = gr.ImageEditor(label='Reviewed mask — edit draft', type='numpy')
    threshold = gr.Slider(0.20, 1.00, value=0.20, step=0.01, label='Display confidence threshold')
    notes = gr.Textbox(label='Review notes')
    save_button = gr.Button('Save reviewed mask and load next', variant='primary')

    threshold.change(update_threshold, inputs=[threshold], outputs=[confidence_view])
    save_button.click(
        save_and_next,
        inputs=[mask_editor, notes],
        outputs=[observed_view, confidence_view, mask_editor, status, notes],
    )

    initial = load_review_item(0)
    demo.load(lambda: initial, outputs=[observed_view, confidence_view, mask_editor, status])

demo.launch(share=False, debug=False)

## Stop after review

Complete the paired reviews, then return to Colab and run the remaining cells. The next cell refuses to close the review lock until every paired page has a reviewed mask and review audit.

In [ ]:
# ============================================================
# 7. Close reviewed-mask lock before resolving clean truth
# ============================================================

lock_rows = []
missing = []

for _, r in roster.iterrows():
    page = str(r.page)
    reviewed = REVIEWED_MASK_DIR / f'{page}_reviewed.png'
    audit = REVIEW_AUDIT_DIR / f'{page}.json'
    draft = DRAFT_MASK_DIR / f'{page}_draft.png'
    conf = CONFIDENCE_DIR / f'{page}_confidence.npy'

    if not (reviewed.exists() and audit.exists() and draft.exists() and conf.exists()):
        missing.append(page)
        continue

    value = json.loads(audit.read_text())
    require(value.get('population_role') == 'PAIRED_HITL_DEVELOPMENT', f'{page}: wrong population role')
    require(value.get('clean_truth_available_to_reviewer') is False, f'{page}: truth-firewall failure')

    lock_rows.append({
        'page': page,
        'annotated_path': str(r.annotated_path),
        'annotated_md5': str(r.annotated_md5),
        'draft_mask': str(draft),
        'draft_mask_sha256': sha256_file(draft),
        'confidence_map': str(conf),
        'confidence_map_sha256': sha256_file(conf),
        'reviewed_mask': str(reviewed),
        'reviewed_mask_sha256': sha256_file(reviewed),
        'review_audit': str(audit),
        'review_audit_sha256': sha256_file(audit),
    })

require(not missing, f'Review incomplete for {len(missing)} pages: {missing[:20]}')
lock_manifest = pd.DataFrame(lock_rows).sort_values('page').reset_index(drop=True)
lock_manifest_path = LOCK_DIR / 'paired_hitl_review_manifest.csv'
lock_manifest.to_csv(lock_manifest_path, index=False)

review_lock = {
    'state': 'CLOSED_BEFORE_CLEAN_TRUTH',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'population_role': 'PAIRED_HITL_DEVELOPMENT',
    'pages': int(len(lock_manifest)),
    'reviewer_code': REVIEWER_CODE,
    'yolo_model_path': str(MODEL_PATH),
    'yolo_model_md5': MODEL_MD5,
    'prediction_confidence': PREDICTION_CONFIDENCE,
    'reviewed_mask_manifest_sha256': sha256_file(lock_manifest_path),
    'clean_truth_read': False,
}
review_lock_path = LOCK_DIR / 'paired_hitl_review_lock.json'
review_lock_path.write_text(json.dumps(review_lock, indent=2))
print(json.dumps(review_lock, indent=2))

## Stage C — Generate and lock A1 and cleanup candidates

This stage still does not resolve clean truth. It loads the actual reviewed experimental masks, frozen YOLO confidence maps and frozen V8.9 operator, constructs A1, and applies the three predeclared cleanup rules.

In [ ]:
# ============================================================
# 8. Load frozen V8.9 operator
# ============================================================

import importlib.util, sys

require(V89_OPERATOR_PATH.exists(), f'V8.9 operator not found: {V89_OPERATOR_PATH}')
spec = importlib.util.spec_from_file_location('v89_notebook87', V89_OPERATOR_PATH)
mod = importlib.util.module_from_spec(spec)
sys.modules['v89_notebook87'] = mod
spec.loader.exec_module(mod)

op = mod.load_v89_operator(
    p_print_bundle_root=P_PRINT_BUNDLE,
    remove_bundle_root=REMOVE_BUNDLE,
)
require(isinstance(op, mod.V89Operator), f'Expected V89Operator, got {type(op)}')
print('Loaded:', type(op).__name__)

In [ ]:
# ============================================================
# 9. Candidate builder using production-style reviewed masks
# ============================================================


def candidate_arrays(observed, seed, conf, page):
    result = op.run_page(observed, seed, page=page)

    prepared = mod.prepare_mask(seed) > 0
    telea = mod.telea_reconstruct(observed, prepared)
    baseline_v78, routes = mod.geometry_t80(observed, prepared, telea)

    require(np.array_equal(baseline_v78, result.baseline_v78), f'{page}: V7.8 baseline mismatch')
    ambiguous = np.asarray(routes['ambiguous'], bool)
    paper = np.asarray(routes['paper'], np.uint8)
    p_print = np.asarray(result.p_print, float)

    a1 = np.asarray(result.corrected, np.uint8).copy()
    a1[ambiguous] = observed[ambiguous]

    arms = {'A1_BASELINE': a1}
    masks = {}
    for name, rule in CANDIDATES.items():
        if rule is None:
            continue
        annotation_floor, print_ceiling = rule
        candidate_mask = (
            ambiguous &
            (conf >= annotation_floor) &
            np.isfinite(p_print) &
            (p_print < print_ceiling)
        )
        candidate = a1.copy()
        candidate[candidate_mask] = paper[candidate_mask]
        require(np.array_equal(candidate[~candidate_mask], a1[~candidate_mask]), f'{page}/{name}: non-candidate regression')
        arms[name] = candidate
        masks[name] = candidate_mask

    return {
        'result': result,
        'prepared': prepared,
        'ambiguous': ambiguous,
        'paper': paper,
        'p_print': p_print,
        'arms': arms,
        'masks': masks,
    }

In [ ]:
# ============================================================
# 10. Generate/resume observable candidate outputs
# ============================================================

review_manifest = pd.read_csv(LOCK_DIR / 'paired_hitl_review_manifest.csv', dtype=str)
page_rows = []

for _, r in review_manifest.iterrows():
    page = str(r.page)
    d = RUN / 'pages' / page
    expected = [d/'observed.png', d/'reviewed_seed.png', d/'ambiguous.png', d/'arrays.npz'] + [d/f'{name}.png' for name in CANDIDATES]

    if all(p.exists() for p in expected):
        print(page, 'REUSE')
        with np.load(d/'arrays.npz') as zz:
            ambiguous_pixels = int((zz['ambiguous'] > 0).sum())
            candidate_counts = {name: int((zz[f'mask__{name}'] > 0).sum()) for name in CANDIDATES if name != 'A1_BASELINE'}
        page_rows.append({'page': page, 'status': 'REUSED', 'ambiguous_pixels': ambiguous_pixels, **{f'{k}_pixels':v for k,v in candidate_counts.items()}})
        continue

    if d.exists():
        raise RuntimeError(f'{page}: partial output directory exists: {d}')

    obs = read_gray(r.annotated_path)
    seed = read_gray(r.reviewed_mask)
    conf = np.load(r.confidence_map).astype(np.float32)
    require(obs.shape == seed.shape == conf.shape, f'{page}: input shape mismatch')

    z = candidate_arrays(obs, seed, conf, page)
    d.mkdir(parents=True)
    cv2.imwrite(str(d/'observed.png'), obs)
    cv2.imwrite(str(d/'reviewed_seed.png'), seed)
    cv2.imwrite(str(d/'ambiguous.png'), z['ambiguous'].astype(np.uint8)*255)
    for name, img in z['arms'].items():
        cv2.imwrite(str(d/f'{name}.png'), img)

    arrays = {
        'ambiguous': z['ambiguous'].astype(np.uint8),
        'p_print': z['p_print'].astype(np.float32),
        'p_annotation': conf.astype(np.float32),
    }
    for name, mask in z['masks'].items():
        arrays[f'mask__{name}'] = mask.astype(np.uint8)
    np.savez_compressed(d/'arrays.npz', **arrays)

    page_rows.append({
        'page': page,
        'status': 'PROCESSED',
        'ambiguous_pixels': int(z['ambiguous'].sum()),
        **{f'{name}_pixels': int(mask.sum()) for name,mask in z['masks'].items()},
    })
    print(page, 'PROCESSED')
    del z
    gc.collect()

observable_summary = pd.DataFrame(page_rows)
observable_summary.to_csv(RUN/'tables'/'observable_candidate_summary.csv', index=False)

In [ ]:
# ============================================================
# 11. Close candidate prediction lock before clean truth
# ============================================================

hash_rows = []
for p in sorted((RUN/'pages').rglob('*')):
    if p.is_file():
        hash_rows.append({'path': str(p.relative_to(RUN)), 'bytes': p.stat().st_size, 'sha256': sha256_file(p)})
hashes = pd.DataFrame(hash_rows)
hashes.to_csv(RUN/'audit'/'prediction_hashes.csv', index=False)

prediction_lock = {
    'state': 'CLOSED_BEFORE_CLEAN_TRUTH',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'population_role': 'PAIRED_HITL_DEVELOPMENT',
    'pages': int(len(review_manifest)),
    'candidates': CANDIDATES,
    'review_lock_sha256': sha256_file(LOCK_DIR/'paired_hitl_review_lock.json'),
    'hash_table_sha256': hashlib.sha256(hashes.sort_values('path').to_csv(index=False).encode()).hexdigest(),
    'v89_operator_sha256': sha256_file(V89_OPERATOR_PATH),
    'clean_truth_read': False,
}
(RUN/'prediction_lock.json').write_text(json.dumps(prediction_lock, indent=2))
print(json.dumps(prediction_lock, indent=2))

## Stage D — Reveal clean truth and score the predeclared candidates

Only this stage resolves `source/clean`. Metrics remain separate for genuine-print loss and annotation cleanup. No automatic winner is selected.

In [ ]:
# ============================================================
# 12. Resolve clean counterparts after both locks are closed
# ============================================================

require(json.loads((LOCK_DIR/'paired_hitl_review_lock.json').read_text())['state'] == 'CLOSED_BEFORE_CLEAN_TRUTH', 'Review lock not closed')
require(json.loads((RUN/'prediction_lock.json').read_text())['state'] == 'CLOSED_BEFORE_CLEAN_TRUTH', 'Prediction lock not closed')

clean_map = {p.stem: p for p in image_files(CLEAN_DIR)}
missing_clean = sorted(set(review_manifest.page.astype(str)) - set(clean_map))
require(not missing_clean, f'Missing clean counterparts: {missing_clean}')
print('Clean truth resolved after locks:', len(clean_map))

In [ ]:
# ============================================================
# 13. Truth scoring
# ============================================================


def printed_ink(clean):
    return cv2.adaptiveThreshold(
        clean, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        41, 15,
    ) > 0

score_rows = []
pixel_parts = []

for _, r in review_manifest.iterrows():
    page = str(r.page)
    d = RUN/'pages'/page
    obs = read_gray(r.annotated_path)
    clean = read_gray(clean_map[page])
    seed = read_gray(r.reviewed_mask)
    require(obs.shape == clean.shape == seed.shape, f'{page}: truth shape mismatch')

    clean_print = printed_ink(clean)
    clean_tol = cv2.dilate(clean_print.astype(np.uint8), np.ones((3,3),np.uint8)) > 0

    with np.load(d/'arrays.npz') as zz:
        ambiguous = zz['ambiguous'] > 0
        p_print = zz['p_print'].astype(float)
        p_annotation = zz['p_annotation'].astype(float)
        masks = {name: zz[f'mask__{name}'] > 0 for name in CANDIDATES if name != 'A1_BASELINE'}

    truth_print = ambiguous & clean_tol
    truth_annotation = ambiguous & (obs < 200) & ~clean_tol
    truth_uncertain = ambiguous & ~truth_print & ~truth_annotation

    for name in CANDIDATES:
        img = read_gray(d/f'{name}.png')
        if name == 'A1_BASELINE':
            cleanup = np.zeros(obs.shape,bool)
        else:
            cleanup = masks[name]
        score_rows.append({
            'page': page,
            'arm': name,
            'ambiguous_pixels': int(ambiguous.sum()),
            'cleanup_pixels': int(cleanup.sum()),
            'truth_annotation_cleaned': int((cleanup & truth_annotation).sum()),
            'truth_print_lost': int((cleanup & truth_print).sum()),
            'truth_uncertain_changed': int((cleanup & truth_uncertain).sum()),
            'annotation_population': int(truth_annotation.sum()),
            'print_population': int(truth_print.sum()),
            'annotation_cleanup_rate': float((cleanup & truth_annotation).sum()/max(truth_annotation.sum(),1)),
            'print_loss_rate': float((cleanup & truth_print).sum()/max(truth_print.sum(),1)),
        })

    ys,xs = np.where(ambiguous)
    if len(ys):
        part = pd.DataFrame({
            'page': page, 'y': ys, 'x': xs,
            'p_annotation': p_annotation[ys,xs],
            'p_print': p_print[ys,xs],
            'truth_print': truth_print[ys,xs],
            'truth_annotation': truth_annotation[ys,xs],
            'truth_uncertain': truth_uncertain[ys,xs],
        })
        for name,mask in masks.items():
            part[f'{name}_selected'] = mask[ys,xs]
        pixel_parts.append(part)

scores = pd.DataFrame(score_rows)
scores.to_csv(RUN/'tables'/'truth_scores_by_page.csv', index=False)
pixels = pd.concat(pixel_parts, ignore_index=True) if pixel_parts else pd.DataFrame()
pixels.to_parquet(RUN/'tables'/'paired_hitl_ambiguous_truth.parquet', index=False)

pooled = scores.groupby('arm',as_index=False).agg(
    pages=('page','nunique'),
    cleanup_pixels=('cleanup_pixels','sum'),
    truth_annotation_cleaned=('truth_annotation_cleaned','sum'),
    truth_print_lost=('truth_print_lost','sum'),
    truth_uncertain_changed=('truth_uncertain_changed','sum'),
    annotation_population=('annotation_population','sum'),
    print_population=('print_population','sum'),
)
pooled['annotation_cleanup_rate'] = pooled.truth_annotation_cleaned / pooled.annotation_population.clip(lower=1)
pooled['print_loss_rate'] = pooled.truth_print_lost / pooled.print_population.clip(lower=1)
pooled['annotation_per_print_lost'] = pooled.truth_annotation_cleaned / pooled.truth_print_lost.clip(lower=1)
pooled.to_csv(RUN/'tables'/'truth_scores_pooled.csv', index=False)
display(pooled)

In [ ]:
# ============================================================
# 14. Page-clustered bootstrap
# ============================================================

BOOTSTRAP_RESAMPLES = 10_000
rng = np.random.default_rng(RANDOM_SEED)
bootstrap_rows = []

for arm, group in scores.groupby('arm'):
    if arm == 'A1_BASELINE':
        continue
    rows = group.reset_index(drop=True)
    ann = rows.annotation_cleanup_rate.to_numpy(float)
    prn = rows.print_loss_rate.to_numpy(float)
    ann_means = np.empty(BOOTSTRAP_RESAMPLES)
    prn_means = np.empty(BOOTSTRAP_RESAMPLES)
    for i in range(BOOTSTRAP_RESAMPLES):
        idx = rng.integers(0, len(rows), len(rows))
        ann_means[i] = ann[idx].mean()
        prn_means[i] = prn[idx].mean()
    bootstrap_rows.append({
        'arm': arm,
        'pages': len(rows),
        'mean_annotation_cleanup_rate': float(ann.mean()),
        'annotation_ci95_low': float(np.quantile(ann_means,0.025)),
        'annotation_ci95_high': float(np.quantile(ann_means,0.975)),
        'mean_print_loss_rate': float(prn.mean()),
        'print_loss_ci95_low': float(np.quantile(prn_means,0.025)),
        'print_loss_ci95_high': float(np.quantile(prn_means,0.975)),
        'pages_with_print_loss': int((rows.truth_print_lost>0).sum()),
    })

bootstrap = pd.DataFrame(bootstrap_rows)
bootstrap.to_csv(RUN/'tables'/'page_clustered_bootstrap.csv', index=False)
display(bootstrap)

In [ ]:
# ============================================================
# 15. Draft-to-reviewed morphology and reviewer intervention
# ============================================================

review_change_rows = []
for _, r in review_manifest.iterrows():
    page = str(r.page)
    draft = read_gray(r.draft_mask) > 0
    reviewed = read_gray(r.reviewed_mask) > 0
    added = reviewed & ~draft
    erased = draft & ~reviewed
    retained = reviewed & draft
    review_change_rows.append({
        'page': page,
        'draft_pixels': int(draft.sum()),
        'reviewed_pixels': int(reviewed.sum()),
        'added_pixels': int(added.sum()),
        'erased_pixels': int(erased.sum()),
        'retained_draft_pixels': int(retained.sum()),
        'reviewed_to_draft_ratio': float(reviewed.sum()/max(draft.sum(),1)),
        'added_fraction_of_reviewed': float(added.sum()/max(reviewed.sum(),1)),
        'erased_fraction_of_draft': float(erased.sum()/max(draft.sum(),1)),
    })
review_changes = pd.DataFrame(review_change_rows)
review_changes.to_csv(RUN/'tables'/'draft_to_reviewed_changes.csv', index=False)
display(review_changes.describe())

In [ ]:
# ============================================================
# 16. Truth atlas and visual summaries
# ============================================================

VIS = RUN/'visuals'
VIS.mkdir(exist_ok=True)

# Frontier
fig, ax = plt.subplots(figsize=(9,7))
for _, row in bootstrap.iterrows():
    ax.scatter(row.mean_print_loss_rate*100, row.mean_annotation_cleanup_rate*100, s=90)
    ax.text(row.mean_print_loss_rate*100, row.mean_annotation_cleanup_rate*100, ' '+row.arm, va='center')
ax.set_xlabel('Mean genuine-print loss (%)')
ax.set_ylabel('Mean additional annotation cleanup (%)')
ax.set_title('Production-profile paired HITL A1 cleanup frontier')
ax.grid(True, alpha=.3)
fig.tight_layout()
fig.savefig(VIS/'paired_hitl_cleanup_frontier.png', dpi=180)
plt.close(fig)

# Rank pages by print loss, then cleanup opportunity.
rank = scores[scores.arm!='A1_BASELINE'].copy()
rank['risk_score'] = rank.truth_print_lost*1000 + rank.truth_annotation_cleaned
rank = rank.sort_values('risk_score', ascending=False)
rank.to_csv(RUN/'tables'/'truth_atlas_review_queue.csv', index=False)

# Create one contact sheet per top 24 page/arm rows.
def rgb(gray): return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)

def overlay_mask(gray, mask, color, alpha=.55):
    b = rgb(gray); o=b.copy(); m=np.asarray(mask,bool); c=np.array(color,np.float32)
    o[m]=np.clip((1-alpha)*b[m]+alpha*c,0,255).astype(np.uint8); return o

for rank_index, row in enumerate(rank.head(24).itertuples(), start=1):
    page = str(row.page); arm=str(row.arm); d=RUN/'pages'/page
    obs=read_gray(review_manifest.set_index('page').loc[page,'annotated_path'])
    clean=read_gray(clean_map[page]); a1=read_gray(d/'A1_BASELINE.png'); cand=read_gray(d/f'{arm}.png')
    with np.load(d/'arrays.npz') as zz:
        p_ann=zz['p_annotation'].astype(float); p_pr=zz['p_print'].astype(float); selection=zz[f'mask__{arm}']>0
    clean_print=printed_ink(clean); clean_tol=cv2.dilate(clean_print.astype(np.uint8),np.ones((3,3),np.uint8))>0
    print_loss=selection & clean_tol
    annotation_gain=selection & (obs<200) & ~clean_tol
    fig,axes=plt.subplots(2,4,figsize=(16,11))
    panels=[
        (rgb(obs),'Annotated'),(rgb(clean),'Clean truth'),(rgb(a1),'A1'),(rgb(cand),arm),
        (overlay_mask(obs,selection,(255,255,0)),'Selected'),
        (overlay_mask(obs,annotation_gain,(0,255,0)),'Truth annotation cleaned'),
        (overlay_mask(obs,print_loss,(255,0,0)),'Truth print lost'),
        (cv2.cvtColor(cv2.applyColorMap(np.clip(p_ann*255,0,255).astype(np.uint8),cv2.COLORMAP_TURBO),cv2.COLOR_BGR2RGB),'p(annotation)'),
    ]
    for ax,(im,title) in zip(axes.flat,panels): ax.imshow(im,cmap='gray');ax.set_title(title);ax.axis('off')
    fig.suptitle(f'{rank_index}. {page} / {arm} — annotation cleaned {row.truth_annotation_cleaned:,}; print lost {row.truth_print_lost:,}')
    fig.tight_layout()
    fig.savefig(VIS/f'truth_atlas_{rank_index:02d}_{page}_{arm}.png',dpi=140)
    plt.close(fig)

print('Visuals:', VIS)

In [ ]:
# ============================================================
# 17. Final descriptive verdict and bundle
# ============================================================

best = None
if len(bootstrap):
    # Descriptive only: lowest upper print-loss bound, then greatest cleanup.
    best = bootstrap.sort_values(['print_loss_ci95_high','mean_annotation_cleanup_rate'], ascending=[True,False]).iloc[0].to_dict()

verdict = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'notebook': '87_Paired_HITL_Development_Truth_Transport',
    'population_role': 'PAIRED_HITL_DEVELOPMENT',
    'review_lock_state': 'CLOSED_BEFORE_CLEAN_TRUTH',
    'prediction_lock_state': 'CLOSED_BEFORE_CLEAN_TRUTH',
    'pages': int(len(review_manifest)),
    'candidates': CANDIDATES,
    'descriptive_best_by_lowest_upper_print_loss': best,
    'automatic_promotion': False,
    'independent_validation': False,
    'verdict': 'DIAGNOSTIC_COMPLETE_NO_AUTOMATIC_WINNER',
}
(RUN/'notebook87_verdict.json').write_text(json.dumps(verdict, indent=2))

summary = [
    'NOTEBOOK 87 — PAIRED HITL DEVELOPMENT TRUTH TRANSPORT',
    '',
    f'Pages reviewed and scored: {len(review_manifest)}',
    'Population: PAIRED_HITL_DEVELOPMENT',
    'Clean truth was unavailable during review and candidate generation.',
    '',
    'Candidates:',
] + [f'  {k}: {v}' for k,v in CANDIDATES.items()] + [
    '',
    'No automatic promotion is authorised.',
    'Existing paired pages are development-exposed and cannot provide final independent promotion evidence.',
]
(RUN/'executive_summary.txt').write_text('
'.join(summary))

upload = RUN/'upload'
for p in [
    RUN/'executive_summary.txt', RUN/'notebook87_verdict.json',
    RUN/'tables'/'truth_scores_pooled.csv', RUN/'tables'/'page_clustered_bootstrap.csv',
    RUN/'tables'/'draft_to_reviewed_changes.csv', RUN/'tables'/'truth_atlas_review_queue.csv',
    VIS/'paired_hitl_cleanup_frontier.png', LOCK_DIR/'paired_hitl_review_lock.json',
    RUN/'prediction_lock.json',
]:
    require(p.exists(), f'Missing bundle artefact: {p}')
    shutil.copy2(p, upload/p.name)

manifest=[]
for p in sorted(upload.iterdir()):
    if p.is_file(): manifest.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(upload/'upload_manifest.csv',index=False)
zip_base=RUN/'notebook87_paired_hitl_truth_transport_bundle'
shutil.make_archive(str(zip_base),'zip',upload)
print('Bundle:',str(zip_base)+'.zip')
print(json.dumps(verdict,indent=2))